In [11]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

import xgboost as xgb

from model import Regressor
import torch
import torch.nn as nn
import torch.optim as optim

from tqdm import tqdm

In [12]:
cta_df = pd.read_parquet('../feature_engineer/output/cta_ridership_with_features.parquet')
cta_df = cta_df.reset_index(drop=True)

In [13]:
cta_df.head(10)

,station_id,stationname,date,daytype,rides,map_id,red,blue,g,brn,...,o,location,lat,lon,line,year,month,day,day_of_week_num,day_of_week_name
0,40350,UIC-Halsted,2001-01-01,U,273,40350,False,True,False,False,...,False,"{""latitude"":""41.875474"",""longitude"":""-87.64970...",41.875474,-87.649707,blue,2001,1,1,0,Monday
1,41130,Halsted-Orange,2001-01-01,U,306,41130,False,False,False,False,...,True,"{""latitude"":""41.84678"",""longitude"":""-87.648088...",41.846780,-87.648088,orange,2001,1,1,0,Monday
2,40760,Granville,2001-01-01,U,1059,40760,True,False,False,False,...,False,"{""latitude"":""41.993664"",""longitude"":""-87.65920...",41.993664,-87.659202,red,2001,1,1,0,Monday
3,40070,Jackson/Dearborn,2001-01-01,U,649,40070,False,True,False,False,...,False,"{""latitude"":""41.878183"",""longitude"":""-87.62929...",41.878183,-87.629296,blue,2001,1,1,0,Monday
4,40090,Damen-Brown,2001-01-01,U,411,40090,False,False,False,True,...,False,"{""latitude"":""41.966286"",""longitude"":""-87.67863...",41.966286,-87.678639,brown,2001,1,1,0,Monday
5,40590,Damen/Milwaukee,2001-01-01,U,870,40590,False,True,False,False,...,False,"{""latitude"":""41.909744"",""longitude"":""-87.67743...",41.909744,-87.677437,blue,2001,1,1,0,Monday
6,40720,East 63rd-Cottage Grove,2001-01-01,U,391,40720,False,False,True,False,...,False,"{""latitude"":""41.780309"",""longitude"":""-87.60585...",41.780309,-87.605857,green,2001,1,1,0,Monday
7,41260,Austin-Lake,2001-01-01,U,399,41260,False,False,True,False,...,False,"{""latitude"":""41.887293"",""longitude"":""-87.77413...",41.887293,-87.774135,green,2001,1,1,0,Monday
8,40230,Cumberland,2001-01-01,U,788,40230,False,True,False,False,...,False,"{""latitude"":""41.984246"",""longitude"":""-87.83802...",41.984246,-87.838028,blue,2001,1,1,0,Monday
9,41120,35-Bronzeville-IIT,2001-01-01,U,448,41120,False,False,True,False,...,False,"{""latitude"":""41.831677"",""longitude"":""-87.62582...",41.831677,-87.625826,green,2001,1,1,0,Monday


# Pre-processing

## Encode categorical features

In [14]:
ordinal_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
onehot_encoder = OneHotEncoder()
scaler = StandardScaler()

preprocessor = make_column_transformer(
    # (ordinal_encoder, make_column_selector(dtype_include=object)),
    (onehot_encoder, make_column_selector(dtype_include=object)),
    (scaler, make_column_selector(dtype_include='number')),
    remainder='passthrough'
)

In [15]:
X = preprocessor.fit_transform(cta_df[['line', 'year', 'month', 'day', 'day_of_week_num', 'day_of_week_name', 'lat', 'lon']])
y = cta_df['rides']

# Train-test split

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Models

## OLS

In [18]:
mod_ols = LinearRegression()
mod_ols.fit(X_train, y_train)
y_pred_ols = mod_ols.predict(X_test)
print("R squared: ", r2_score(y_test, y_pred_ols))

R squared:  0.21970786939378295


## RF

In [28]:
mod_rf = RandomForestRegressor(n_estimators=10, random_state=42)
mod_rf.fit(X_train, y_train)
y_pred_rf = mod_rf.predict(X_test)
print("R squared: ", r2_score(y_test, y_pred_rf))

R squared:  0.9690758661766381


## XGBoost

In [20]:
mod_xgb = xgb.XGBRegressor()
mod_xgb.fit(X_train, y_train)
y_pred_xgb = mod_xgb.predict(X_test)
print("R squared: ", r2_score(y_test, y_pred_xgb))

R squared:  0.9521481990814209


## Neural network using PyTorch

In [23]:
torch.manual_seed(42)

# Preprocess 
y_train_scaled = scaler.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler.transform(y_test.values.reshape(-1, 1))

# Initialize
model = Regressor(n_in=X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train
X_train_tensor = torch.from_numpy(X_train).float()
y_train_tensor = torch.from_numpy(y_train_scaled).float()

for epoch in tqdm(range(500)):
    optimizer.zero_grad()
    preds = model(X_train_tensor)
    loss = criterion(preds, y_train_tensor)
    loss.backward()
    optimizer.step()

# Evaluate
model.eval()
X_test_tensor = torch.from_numpy(X_test).float()
y_test_tensor = torch.from_numpy(y_test.values.astype(float)).float()

with torch.no_grad():
    y_pred_scaled = model(X_test_tensor)
    y_pred = scaler.inverse_transform(y_pred_scaled.numpy())
    r2_nn = r2_score(y_test_tensor.numpy(), y_pred)
    print("R squared: ", r2_nn)

100%|██████████| 500/500 [04:35<00:00,  1.82it/s]


R squared:  0.4870026707649231
